# Run Model Comparison

Interactive version of `scripts/compare_models.py` — same sampling/adapter/
scoring logic (imported, not duplicated), broken into cells so you can
change config and re-run without reloading models each time. The CLI script
still exists too, for unattended batch runs (e.g. an overnight 300-image
pass across every model) where a notebook you have to babysit is worse than
`nohup python scripts/compare_models.py ... &`.

Writes results to `data/comparisons/<run_name>/` — open `ModelComparison.ipynb`
afterward to score and visualize them.

Run from the repo root.


In [ ]:
import json
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from types import SimpleNamespace

import pandas as pd

REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from compare_models import COMPARISONS_ROOT, DEFAULT_YOLO_WEIGHTS, build_adapter, image_path_for, sample_test_images  # noqa: E402
from model_adapters import ADAPTERS  # noqa: E402

## Configuration — edit these and re-run

In [ ]:
N_IMAGES = 200
MODELS = ["yolo", "florence2"]  # available: yolo, florence2, ollama, claude
INCLUDE_CLOUD = False  # must be True to allow a cloud model (claude) to run —
                       # guards against accidental API spend, especially from
                       # a stray "Run All"
SEED = 42
RUN_NAME = None  # None = auto timestamp

model_config = SimpleNamespace(
    yolo_weights=str(DEFAULT_YOLO_WEIGHTS),
    florence2_model="microsoft/Florence-2-base",
    ollama_model="llava",
    ollama_url="http://localhost:11434",
    claude_model="claude-opus-5",
)

## Sample test images

In [ ]:
unknown = [m for m in MODELS if m not in ADAPTERS]
if unknown:
    raise ValueError(f"Unknown model(s) {unknown}. Available: {list(ADAPTERS)}")

cloud_requested = [m for m in MODELS if ADAPTERS[m][1]]
if cloud_requested and not INCLUDE_CLOUD:
    raise ValueError(
        f"{cloud_requested} require INCLUDE_CLOUD = True (network calls to a paid API) — "
        "set it above and re-run this cell to confirm."
    )

sampled_files = sample_test_images(N_IMAGES, SEED)
print(f"Sampled {len(sampled_files)} test images (seed={SEED})")

## Load models

In [ ]:
adapters = {}
for name in MODELS:
    print(f"Loading {name}...")
    t0 = time.perf_counter()
    adapters[name] = build_adapter(name, model_config)
    print(f"  ready in {time.perf_counter() - t0:.1f}s")

## Run predictions

This is the slow cell — Florence-2 is ~3-5s/image (one call per positive
class), cloud models add network latency on top. Progress prints every 20
(image, model) pairs.

In [ ]:
detection_rows = []
presence_rows = []
parse_failures = {name: 0 for name in MODELS}

total = len(sampled_files) * len(MODELS)
done = 0
t_start = time.perf_counter()

for file_stem in sampled_files:
    image_path = image_path_for(file_stem)
    if image_path is None:
        print(f"warning: no image found for {file_stem}, skipping")
        continue

    for name, adapter in adapters.items():
        done += 1
        detections = adapter.predict(image_path)

        if detections is None:  # unparseable model output
            parse_failures[name] += 1
            for cls in adapter.queryable_classes:
                presence_rows.append(
                    {"file": file_stem, "model": name, "class_name": cls, "present": None, "parse_error": True}
                )
            continue

        present_classes = {d.class_name for d in detections if d.present}
        for cls in adapter.queryable_classes:
            presence_rows.append(
                {
                    "file": file_stem,
                    "model": name,
                    "class_name": cls,
                    "present": cls in present_classes,
                    "parse_error": False,
                }
            )
        for d in detections:
            if d.bbox is None and not d.present:
                continue  # a chat-model "false" isn't a detection row
            detection_rows.append(
                {
                    "file": file_stem,
                    "model": name,
                    "class_name": d.class_name,
                    "confidence": d.confidence,
                    "x1": d.bbox[0] if d.bbox else None,
                    "y1": d.bbox[1] if d.bbox else None,
                    "x2": d.bbox[2] if d.bbox else None,
                    "y2": d.bbox[3] if d.bbox else None,
                }
            )

        if done % 20 == 0 or done == total:
            elapsed = time.perf_counter() - t_start
            print(f"  {done}/{total} (image, model) pairs done, {elapsed:.0f}s elapsed")

print(f"\nDone in {time.perf_counter() - t_start:.0f}s")

## Save results

In [ ]:
run_name = RUN_NAME or datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
run_dir = COMPARISONS_ROOT / run_name
run_dir.mkdir(parents=True, exist_ok=True)

detections_path = run_dir / "detections.csv"
presence_path = run_dir / "presence.csv"
pd.DataFrame(detection_rows).to_csv(detections_path, index=False)
pd.DataFrame(presence_rows).to_csv(presence_path, index=False)

manifest = {
    "run_name": run_name,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "n_images_requested": N_IMAGES,
    "n_images_sampled": len(sampled_files),
    "seed": SEED,
    "models": MODELS,
    "queryable_classes": {name: adapters[name].queryable_classes for name in MODELS},
    "supports_grounding": {name: adapters[name].supports_grounding for name in MODELS},
    "config": vars(model_config),
    "parse_failures": parse_failures,
    "sampled_files": sampled_files,
}
manifest_path = run_dir / "run_manifest.json"
with manifest_path.open("w") as f:
    json.dump(manifest, f, indent=2)

print(f"Wrote:\n  {manifest_path}\n  {detections_path} ({len(detection_rows)} rows)\n  {presence_path} ({len(presence_rows)} rows)")
if any(parse_failures.values()):
    print(f"parse failures: {parse_failures}")
print(f"\nOpen ModelComparison.ipynb (RUN_NAME = {run_name!r}, or leave None for latest) to score and visualize.")